# FedFlow-LLM v2: Colab training run (Telecom Italia, Milan data)

v2 is defined in `PLAN_V2.md` on branch `v2`: v1 plus (a) loss on answer tokens only, (b) the Milan time and day type of the
last history value in the prompt, (c) onset-balanced sampling of the training examples. Same model, LoRA settings, 25 rounds,
6 clients, seeds and split.

**The branch `v2` must be pushed to GitHub before you run this notebook** (`git push -u origin v2` on your own machine),
because the clone cell switches to it.

Run cells top to bottom, one at a time. Never run a cell twice. Use a GPU runtime (T4).

There is **no inference cell**: v1 and v2 are both scored locally on the same RTX 4060 (see `V2_RUNBOOK.md`).

**Keep-alive (optional, not a cell):** press F12, open the Console tab on this Colab page, and paste this snippet so Colab doesn't disconnect while it's idle:

```javascript
function keepAlive() {
  // Click the "Connect"/status button every 60 s so Colab sees activity.
  const btn = document.querySelector("colab-connect-button");
  if (btn) btn.shadowRoot.querySelector("#connect").click();
}
setInterval(keepAlive, 60000);
```

In [ ]:
# Confirm a GPU is attached.
!nvidia-smi

In [ ]:
# Install dependencies (order matters, see README).
!pip install -q torch transformers peft accelerate bitsandbytes "flwr[simulation]"
!pip uninstall -y torch
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip uninstall -y torchvision torchaudio
!pip install -q transformers==4.46.3 peft==0.13.2

In [ ]:
# Clone the repo, switch to the v2 branch (it must be pushed first) and move into training/.
!git clone https://github.com/shashank69420/fedflow-llm.git
%cd fedflow-llm
!git switch v2 && git log --oneline -1
%cd training

In [ ]:
# Generate the v2 telecom data (into data/telecom_v2/, 200 examples) and check the split:
# no training/validation/test slot overlap, each client's slice holds only its own cell, prompts carry the time line.
!python generate_data.py --dataset telecom --variant v2 && python check_split.py --dataset telecom --variant v2

client.py reads `data/traffic_train.json`, and infer.py reads `data/val_windows.json` / `data/test_windows.json`. The next cell copies
the four v2 files from `data/telecom_v2/` into `data/`. This is a fresh clone, so nothing else is overwritten.

In [ ]:
# Copy the four v2 files into data/, where client.py and infer.py look, and confirm each copy is identical.
import filecmp
import json
import shutil

for name in ["traffic_train.json", "train_starts.json", "val_windows.json", "test_windows.json"]:
    shutil.copyfile(f"data/telecom_v2/{name}", f"data/{name}")
    assert filecmp.cmp(f"data/telecom_v2/{name}", f"data/{name}", shallow=False), f"copy of {name} differs"
    print("copied", name)

# The validation/test windows must be tagged v2, so infer.py can never pair them with a v1 adapter.
assert all(w.get("variant") == "v2" for w in json.load(open("data/val_windows.json")))
print("training examples:", len(json.load(open("data/traffic_train.json"))))
!ls -l data/

SMOKE TEST: run the next two cells, then STOP and check.

**Pass** = both rounds finish, `outputs/adapter.npz` and `outputs/run_config.json` exist, and there is no
"ABORT: ... longer than MAX_LENGTH" message. Its numbers are not results.

`FEDFLOW_MAX_LENGTH=288` is the v2 limit measured with the real tokenizer (`training/data/telecom_v2/token_audit_v2.txt`:
longest v2 text 272 tokens; smallest multiple of 32 at least 16 above it = 288).

In [ ]:
# Smoke test: v2 training for 2 rounds only.
!FEDFLOW_VARIANT=v2 FEDFLOW_MAX_LENGTH=288 FEDFLOW_ROUNDS=2 python server.py

In [ ]:
# Smoke-test check: the adapter and run_config.json exist and say v2 / 288 / 2 rounds.
import json, os
assert os.path.exists("outputs/adapter.npz"), "outputs/adapter.npz missing: the smoke test FAILED"
assert os.path.exists("outputs/run_config.json"), "outputs/run_config.json missing: the smoke test FAILED"
cfg = json.load(open("outputs/run_config.json"))
print(json.dumps(cfg, indent=2))
assert cfg["variant"] == "v2" and cfg["max_length"] == 288 and cfg["rounds"] == 2
print("SMOKE TEST PASSED")
!ls -lh outputs

DO NOT RUN BELOW until the smoke test has been checked.

In [ ]:
# FULL RUN (attempt 1): 25 rounds of v2 training, 200 examples. The smoke-test outputs are removed first,
# so nothing from the 2-round run can be mistaken for the full run.
!rm -rf outputs && FEDFLOW_VARIANT=v2 FEDFLOW_MAX_LENGTH=288 FEDFLOW_ROUNDS=25 python server.py

In [ ]:
# Run IMMEDIATELY after training: zip outputs/ plus the v2 data used, and download it.
# Contents: outputs/adapter.npz, loss_history.csv, run_config.json, and data/telecom_v2/ (incl. build_report.txt).
import json, os
assert os.path.exists("outputs/adapter.npz") and json.load(open("outputs/run_config.json"))["rounds"] == 25
!rm -f v2_attempt1_outputs.zip
!zip -r v2_attempt1_outputs.zip outputs/ data/telecom_v2/
from google.colab import files
files.download("v2_attempt1_outputs.zip")

# ATTEMPT 2 ONLY IF PLAN_V2 SAYS SO

Run this section **only** if attempt 1 was scored locally and `compare_variants.py` printed **DO NOT ADOPT**.
If attempt 1 was adopted, do not run anything below.

Attempt 2 is identical to attempt 1 except `FEDFLOW_N_EXAMPLES=400`. Nothing else may change (PLAN_V2.md).
Use a fresh runtime: run the GPU check, install and clone cells at the top first, then the cells below.
`FEDFLOW_MAX_LENGTH=288` still fits: the longest of the 400 examples is 272 tokens (token_audit_v2.txt).

In [ ]:
# Attempt 2 data: 400 v2 examples (data/telecom_v2/ is rewritten), then the same split check.
!FEDFLOW_N_EXAMPLES=400 python generate_data.py --dataset telecom --variant v2 && python check_split.py --dataset telecom --variant v2

In [ ]:
# Attempt 2: copy the four v2 files into data/, where client.py and infer.py look, and confirm each copy is identical.
import filecmp
import json
import shutil

for name in ["traffic_train.json", "train_starts.json", "val_windows.json", "test_windows.json"]:
    shutil.copyfile(f"data/telecom_v2/{name}", f"data/{name}")
    assert filecmp.cmp(f"data/telecom_v2/{name}", f"data/{name}", shallow=False), f"copy of {name} differs"
    print("copied", name)

# The validation/test windows must be tagged v2, so infer.py can never pair them with a v1 adapter.
assert all(w.get("variant") == "v2" for w in json.load(open("data/val_windows.json")))
print("training examples:", len(json.load(open("data/traffic_train.json"))))
!ls -l data/
assert len(json.load(open("data/traffic_train.json"))) == 400

In [ ]:
# Attempt 2 FULL RUN: 25 rounds, 400 examples, otherwise identical to attempt 1.
!rm -rf outputs && FEDFLOW_N_EXAMPLES=400 FEDFLOW_VARIANT=v2 FEDFLOW_MAX_LENGTH=288 FEDFLOW_ROUNDS=25 python server.py

In [ ]:
# Run IMMEDIATELY after training: zip and download attempt 2.
import json, os
cfg = json.load(open("outputs/run_config.json"))
assert os.path.exists("outputs/adapter.npz") and cfg["rounds"] == 25 and cfg["n_examples"] == 400
!rm -f v2_attempt2_outputs.zip
!zip -r v2_attempt2_outputs.zip outputs/ data/telecom_v2/
from google.colab import files
files.download("v2_attempt2_outputs.zip")